# R01 — MVTec confirmation under the protocol that the HPO and the surrogate actually used

**GPU recommended (≈ 1–2 h on one GPU for the 24 candidates + 2 reference models; `REVIEW_SMOKE=1` = 2-minute plumbing check).**
Needs the MVTec AD images (`MVTEC_ROOT`, see the configuration cell). Addresses **R1/R2 "single-category confirmation"**
(does the −0.20 gap survive once the pooling problem is removed?), **R2.3** (does HPO beat a no-HPO PatchCore at the *same*
review budget?) and the cross-dataset-comparability request (**R2.5**).

### Why this notebook exists
`R00` shows that the MVTec surrogate was trained on a target produced by **one PatchCore + one review-budget threshold per
category, averaged over four categories** (`mvtec/01_data_collection.ipynb`), whereas the paper's confirmation
(`mvtec/05_confirmation_evaluation.ipynb`) trains **one pooled memory bank and one pooled threshold**. A score scale that
differs between categories makes the pooled F1 near-meaningless (pooled AUROC ≈ 0.50). The two numbers are therefore not
comparable and the reported surrogate error of −0.20 mixes surrogate error with a **protocol change**.

### What is computed (all candidates are the committed 24-row table, nothing is regenerated)
| block | what | answers |
|---|---|---|
| A | HPO-best configuration re-run with its original seed offset | does the pipeline reproduce the logged value and `best_params_final_results.csv`? |
| B | the 24 candidates, **per-category protocol** (= surrogate target) and the pooled view of the *same* models | protocol-matched gap |
| C | gap decomposition: surrogate error (val) → generalisation (test) | what is lost where |
| D | no-HPO reference PatchCore vs HPO-best at **identical review budgets** + perfect/random-ranker bounds | R2.3, budget confound |
| E | prevalence-matched F1 (MVTec re-weighted to Proditec's 34.5 % positive rate) | R2.5 |

In [ ]:
# ============================================================
# C0) Common setup (identical in every review notebook)
# ============================================================
from pathlib import Path
import json, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
try:
    display
except NameError:
    display = print

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 150, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})
SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)

# ------------------------------------------------------------
# Robust repo-root detection (CWD, its parents, then rglob)
# ------------------------------------------------------------
CWD = Path.cwd().resolve()


def _is_repo(p: Path) -> bool:
    return (p / "mvtec" / "outputs_patchcore").is_dir() and (p / "proditec" / "outputs_patchcore").is_dir()


REPO = next((p for p in [CWD, *CWD.parents] if _is_repo(p)), None)
if REPO is None:
    for base in [CWD, CWD.parent]:
        for hit in base.rglob("outputs_patchcore"):
            if _is_repo(hit.parent.parent):
                REPO = hit.parent.parent
                break
        if REPO:
            break
if REPO is None:
    raise FileNotFoundError("Could not locate the repository root (needs mvtec/ and proditec/). "
                            "Launch Jupyter from the repo or from review/.")
REVIEW = REPO / "review"
OUT = REVIEW / "outputs"
OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
for d in (OUT_TABLES, OUT_FIGS):
    d.mkdir(parents=True, exist_ok=True)

DS = {
    "mvtec": dict(root=REPO / "mvtec", pre="mvtec",
                  data="surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv"),
    "proditec": dict(root=REPO / "proditec", pre="proditec",
                     data="surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"),
}
for k, v in DS.items():
    r = v["root"]
    v["art"] = r / "surrogate_pkl_cfs_metadata"
    v["cfs"] = v["art"] / f"{k}_counterfactuals" / f"{k}_all_methods_cfs.csv"
    v["eval"] = v["art"] / f"{k}_counterfactuals_evaluation" / f"{k}_all_methods_cfs_patchcore_evaluation.csv"
    v["rev"] = v["art"] / f"{k}_revision_analysis"
    v["decomp"] = r / "revision_outputs" / "tables" / f"decomposition_per_candidate_{k}.csv"
    v["model"] = v["art"] / f"{k}_surrogate_catboost.cbm"
    v["meta"] = v["art"] / f"{k}_surrogate_metadata.json"
    v["dataset"] = r / v["data"]
    v["confirm"] = r / "revision_outputs" / "tables" / f"baseline_confirmation_{k}.csv"
    v["confirm_cands"] = r / "revision_outputs" / "tables" / f"baseline_confirm_candidates_{k}.csv"
print("REPO  :", REPO)
print("OUT   :", OUT)


def save_table(table: pd.DataFrame, stem: str, ff="%.4f"):
    p = OUT_TABLES / f"{stem}.csv"
    table.to_csv(p, index=False)
    try:
        table.to_latex(OUT_TABLES / f"{stem}.tex", index=False, escape=True, float_format=ff)
    except Exception as e:
        print("  (tex export skipped:", e, ")")
    print("Saved table:", p.relative_to(REPO))
    return p


def save_fig(fig, stem: str):
    for ext in ("pdf", "png"):
        fig.savefig(OUT_FIGS / f"{stem}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("Saved figure:", (OUT_FIGS / f"{stem}.pdf").relative_to(REPO))


def merge_numbers(section: str, values: dict):
    """Append to outputs/frozen_numbers.json (one section per notebook)."""
    p = OUT / "frozen_numbers.json"
    cur = json.loads(p.read_text()) if p.exists() else {}
    cur[section] = json.loads(json.dumps(values, default=float))
    p.write_text(json.dumps(cur, indent=2, sort_keys=True))
    print("Updated", p.relative_to(REPO), "->", section)

In [ ]:
# ============================================================
# C1) GPU / dataset configuration  <-- EDIT THIS CELL
# ============================================================
import os, time, pickle

# Local copies of the datasets (the split CSVs hold absolute paths of the original server and are
# re-rooted onto these folders).  Environment variables take precedence.
#   MVTEC_ROOT : folder that contains bottle/ cable/ capsule/ hazelnut/ (MVTec AD)
#   PDT_ROOT   : folder that contains train/ and test/ (Proditec public dataset; proprietary)
MVTEC_ROOT = os.environ.get("MVTEC_ROOT", "")      # e.g. "/data/mvtec_anomaly_detection"
PDT_ROOT = os.environ.get("PDT_ROOT", "")          # e.g. "/data/pdt/dataset_public"
DEVICE_NAME = os.environ.get("REVIEW_DEVICE", "")  # "" = auto (cuda if available), or "cuda:1" / "cpu"
SMOKE = os.environ.get("REVIEW_SMOKE", "0") == "1" # 1 = tiny run (2-3 items) to check the plumbing
NUM_WORKERS = int(os.environ.get("REVIEW_WORKERS", "2"))

sys.path.insert(0, str(REVIEW))
import torch
import patchcore_review_lib as L
L.NUM_WORKERS = NUM_WORKERS
L.set_device(DEVICE_NAME or None)
torch.manual_seed(SEED)
if SMOKE:   # plumbing runs never touch the real outputs / cache
    OUT = OUT / "smoke"
    OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
    for _d in (OUT_TABLES, OUT_FIGS):
        _d.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "cache"
CACHE.mkdir(exist_ok=True)
print("device:", L.DEVICE, "| smoke:", SMOKE)
print("MVTEC_ROOT:", MVTEC_ROOT or "(unset)", "| PDT_ROOT:", PDT_ROOT or "(unset)")


def need_images(df: pd.DataFrame, what: str):
    rate = L.path_exist_rate(df)
    print(f"{what}: {len(df)} images, path_exists_rate={rate:.3f}")
    if rate < 1.0:
        raise FileNotFoundError(f"{what}: some images are missing. Set MVTEC_ROOT / PDT_ROOT "
                                "(see the configuration cell).")


def load_data(ds: str):
    """ds in {'mvtec' (per-category dict), 'mvtec_pooled' (one pooled frame set), 'proditec'}."""
    if ds.startswith("mvtec"):
        cd = L.load_mvtec_class_data(REPO / "mvtec" / "mvtec_splits", MVTEC_ROOT or None)
        if SMOKE:
            cd = {c: L.smoke_subsample(v, 14) for c, v in cd.items()}
        data = L.pool_class_data(cd) if ds == "mvtec_pooled" else cd
        if ds == "mvtec":
            for c, v in cd.items():
                need_images(pd.concat([v["train"], v["val"], v["final"]]), f"MVTec/{c}")
        else:
            need_images(pd.concat([data["train"], data["val"], data["final"]]), "MVTec pooled")
        return data
    data = L.load_proditec_data(REPO / "proditec" / "patchcore_splits", PDT_ROOT or None)
    if SMOKE:
        data = L.smoke_subsample(data, 40)
    need_images(pd.concat([data["train"], data["val"], data["final"]]), "Proditec")
    return data

In [ ]:
# ============================================================
# R1.1) Data, candidate table, HPO log
# ============================================================
import hashlib
from sklearn.metrics import roc_auc_score

DS_NAME = "mvtec"
S = DS[DS_NAME]
cands = pd.read_csv(S["cfs"])
print("candidate table:", S["cfs"].relative_to(REPO), cands.shape, "| md5", hashlib.md5(S["cfs"].read_bytes()).hexdigest()[:10])
print(cands.groupby("method").agg(n=("cf_id", "size"), surrogate=("surrogate_pred", "mean"), budget=("params_soft_review_budget", "mean")).round(3))

hpo = L.load_hpo_trials(REPO / "mvtec" / "outputs_patchcore" / "mvtec_4class_patchcore_hpo_logs.db")
best_trial = hpo.sort_values("value", ascending=False).iloc[0]
print(f"\nHPO log: {len(hpo)} trials | best logged value {best_trial.value:.4f} (trial {int(best_trial.number)})")
best_params = json.loads((REPO / "mvtec" / "outputs_patchcore" / "best_params.json").read_text())
best_final = pd.read_csv(REPO / "mvtec" / "outputs_patchcore" / "best_params_final_results.csv")

data = load_data("mvtec")                        # {class: {train, val, final}}
prev = {c: float(v["final"].label.mean()) for c, v in data.items()}
print("test prevalence per class:", {k: round(v, 3) for k, v in prev.items()})

if SMOKE:
    cands = cands.head(3).copy()
    print("SMOKE: using", len(cands), "candidates")
SEEDS = (SEED,)                                   # R03 adds seeds; here one seed keeps the cost at 24 x 4 PatchCore fits

In [ ]:
# ============================================================
# R1.2) Block A - re-run the HPO-best configuration with its original seed offset
# ============================================================
bp = dict(best_params)
bp["soft_review_budget"] = float(best_params["soft_review_budget"])
tA, rawA = L.run_many("mvtec", [bp], data, seeds=(SEED,), cache_dir=CACHE, ids=["hpo_best"], offsets=[int(best_trial.number)])
a = tA.iloc[0]
cmpA = pd.DataFrame([
    dict(quantity="mean validation F1 at budget (HPO objective)", logged=float(best_trial.value), rerun=float(a.f1_val)),
])
display(cmpA)
scoresA = rawA[("hpo_best", SEED)]
rowsA = []
for cls in data:
    yt, st = scoresA[cls]["test"]
    thr_self = L.threshold_from_review_budget(st, bp["soft_review_budget"])
    p, r, f = L.prf_at_threshold(yt, st, thr_self)
    ref = best_final[best_final.mvtec_class == cls].iloc[0]
    rowsA.append(dict(category=cls, f1_final_rerun=f, f1_final_committed=float(ref.f1_final_at_budget),
                      auroc_rerun=L.safe_auroc(yt, st), auroc_committed=float(ref.auroc_final)))
tabA = pd.DataFrame(rowsA); tabA["abs_diff_f1"] = (tabA.f1_final_rerun - tabA.f1_final_committed).abs()
display(tabA.round(4))
save_table(tabA, "R01_hpo_best_reproduction")
print(f"\nmax |F1 rerun - committed| = {tabA.abs_diff_f1.max():.4f} ; logged vs rerun validation objective: "
      f"{float(best_trial.value):.4f} vs {float(a.f1_val):.4f}")

In [ ]:
# ============================================================
# R1.3) Block B - the 24 candidates under the per-category (HPO) protocol
# ============================================================
cfgs = cands.to_dict("records")
ids = cands.cf_id.tolist()
tB, rawB = L.run_many("mvtec", cfgs, data, seeds=SEEDS, cache_dir=CACHE, ids=ids, offsets=ids)   # seed_offset = cf_id as in mvtec/05
tB = tB.merge(cands[["cf_id", "method", "surrogate_pred", "distance_to_factual", "params_soft_review_budget"]]
              .rename(columns={"cf_id": "id"}), on="id")
tB["gap_val_minus_surrogate"] = tB.f1_val - tB.surrogate_pred             # surrogate error, same protocol as its target
tB["gap_test_minus_val"] = tB.f1_test - tB.f1_val                        # generalisation to the held-out final split
tB["gap_test_minus_surrogate"] = tB.f1_test - tB.surrogate_pred
tB["pooled_gap_test_minus_surrogate"] = tB.pooled_f1_test - tB.surrogate_pred
cols = ["id", "method", "surrogate_pred", "budget", "f1_val", "f1_test", "pooled_f1_test", "auroc_test", "pooled_auroc_test",
        "gap_val_minus_surrogate", "gap_test_minus_val", "gap_test_minus_surrogate", "pooled_gap_test_minus_surrogate",
        "bottle_f1_test", "cable_f1_test", "capsule_f1_test", "hazelnut_f1_test", "runtime_sec"]
save_table(tB[cols], "R01_candidates_per_category")
summ = (tB.groupby("method").agg(n=("id", "size"), surrogate=("surrogate_pred", "mean"), f1_val=("f1_val", "mean"),
                                 f1_test=("f1_test", "mean"), pooled_f1_test=("pooled_f1_test", "mean"),
                                 auroc_test=("auroc_test", "mean"), pooled_auroc=("pooled_auroc_test", "mean"),
                                 gap_val=("gap_val_minus_surrogate", "mean"), gap_test=("gap_test_minus_surrogate", "mean"),
                                 gap_pooled=("pooled_gap_test_minus_surrogate", "mean")).reset_index())
allr = tB[["surrogate_pred", "f1_val", "f1_test", "pooled_f1_test", "auroc_test", "pooled_auroc_test", "gap_val_minus_surrogate",
           "gap_test_minus_surrogate", "pooled_gap_test_minus_surrogate"]].mean()
summ = pd.concat([summ, pd.DataFrame([dict(method="ALL", n=len(tB), surrogate=allr.surrogate_pred, f1_val=allr.f1_val, f1_test=allr.f1_test,
                  pooled_f1_test=allr.pooled_f1_test, auroc_test=allr.auroc_test, pooled_auroc=allr.pooled_auroc_test,
                  gap_val=allr.gap_val_minus_surrogate, gap_test=allr.gap_test_minus_surrogate, gap_pooled=allr.pooled_gap_test_minus_surrogate)])],
                 ignore_index=True)
display(summ.round(4))
save_table(summ, "R01_method_summary_protocol_matched")
print("\nPaper (pooled protocol, Table 4): confirmed F1 0.569-0.623, AUROC ~0.50, gap about -0.20.")
print(f"This run: per-category test F1 {allr.f1_test:.3f}, per-category AUROC {allr.auroc_test:.3f}, gap vs surrogate {allr.gap_test_minus_surrogate:+.3f}; "
      f"pooled view of the same models: F1 {allr.pooled_f1_test:.3f}, AUROC {allr.pooled_auroc_test:.3f}, gap {allr.pooled_gap_test_minus_surrogate:+.3f}.")

In [ ]:
# ============================================================
# R1.4) Block C - how much of the surrogate error is selection (winner's curse) vs protocol?
# ============================================================
# (1) Rank agreement between surrogate prediction and re-measured validation F1 across the 24 candidates.
from scipy.stats import spearmanr, pearsonr
rho_val = spearmanr(tB.surrogate_pred, tB.f1_val)[0] if len(tB) > 2 else np.nan
rho_test = spearmanr(tB.surrogate_pred, tB.f1_test)[0] if len(tB) > 2 else np.nan
# (2) Logged HPO trials reproduce their logged value (here: the best trial, block A). A mean shift of the *candidates* relative
#     to the surrogate, with the best trial reproduced, isolates selection effects from pipeline differences.
diag = pd.DataFrame([
    dict(quantity="mean surrogate prediction (candidates)", value=tB.surrogate_pred.mean()),
    dict(quantity="mean re-measured validation F1 (per-category protocol)", value=tB.f1_val.mean()),
    dict(quantity="mean held-out final F1 (per-category protocol)", value=tB.f1_test.mean()),
    dict(quantity="mean held-out final F1, pooled protocol (paper)", value=tB.pooled_f1_test.mean()),
    dict(quantity="surrogate error under the surrogate's own protocol (val - pred)", value=tB.gap_val_minus_surrogate.mean()),
    dict(quantity="generalisation shift (final - val), per-category", value=tB.gap_test_minus_val.mean()),
    dict(quantity="protocol effect on F1 (pooled - per-category, same models)", value=(tB.pooled_f1_test - tB.f1_test).mean()),
    dict(quantity="Spearman(surrogate, val F1) over the 24 candidates", value=rho_val),
    dict(quantity="Spearman(surrogate, final F1) over the 24 candidates", value=rho_test),
    dict(quantity="range of re-measured val F1 over candidates (max-min)", value=tB.f1_val.max() - tB.f1_val.min()),
    dict(quantity="range of review budgets over candidates (max-min)", value=tB.budget.max() - tB.budget.min()),
])
display(diag.round(4)); save_table(diag, "R01_gap_decomposition")

In [ ]:
# ============================================================
# R1.5) Block D - HPO-best vs a no-HPO reference at identical review budgets (R2.3) + ranker bounds
# ============================================================
ref_cfg = dict(L.REFERENCE_CFG)
valprev = float(np.mean([v["val"].label.mean() for v in data.values()]))      # budget a priori = prevalence in the labelled validation split
tR, rawR = L.run_many("mvtec", [{**ref_cfg, "soft_review_budget": float(best_params["soft_review_budget"])}], data,
                      seeds=SEEDS, cache_dir=CACHE, ids=["reference"], offsets=[0])
BUDGETS = [0.10, 0.20, 0.30, 0.40, 0.495, 0.60, 0.70, round(valprev, 3)]
BUDGETS = sorted(set(BUDGETS))
sw_ref = L.sweep_budget("mvtec", rawR[("reference", SEED)], BUDGETS); sw_ref.insert(0, "model", "reference (no HPO)")
sw_hpo = L.sweep_budget("mvtec", rawA[("hpo_best", SEED)], BUDGETS); sw_hpo.insert(0, "model", "HPO best")
SW = pd.concat([sw_hpo, sw_ref], ignore_index=True)
SW["f1_val_minus_random"] = SW.f1_val - SW["random"]
display(SW.round(3)); save_table(SW, "R01_hpo_vs_reference_budget_sweep")
cmp = SW.pivot(index="budget", columns="model", values="f1_test")
cmp["HPO - reference"] = cmp["HPO best"] - cmp["reference (no HPO)"]
cmp["perfect ranker"] = sw_hpo.set_index("budget")["perfect"]; cmp["random ranker"] = sw_hpo.set_index("budget")["random"]
display(cmp.round(3)); save_table(cmp.reset_index(), "R01_hpo_minus_reference_at_equal_budget")
at = lambda b: cmp.loc[min(cmp.index, key=lambda x: abs(x - b))]
print(f"\nAt the HPO budget 0.495: HPO {at(0.495)['HPO best']:.3f} vs reference {at(0.495)['reference (no HPO)']:.3f} "
      f"(perfect ranker {at(0.495)['perfect ranker']:.3f}, random {at(0.495)['random ranker']:.3f}).")
print(f"Reference model with the budget set a priori to the validation prevalence ({valprev:.3f}): "
      f"F1 {at(valprev)['reference (no HPO)']:.3f}.")

In [ ]:
# ============================================================
# R1.6) Block E - prevalence-matched F1 and per-category AUROC (cross-dataset comparability, R2.5)
# ============================================================
PDT_PREV = 0.345
rows = []
for name, raw in (("HPO best", rawA[("hpo_best", SEED)]), ("reference (no HPO)", rawR[("reference", SEED)])):
    for cls, v in raw.items():
        yv, sv = v["val"]; yt, st = v["test"]
        thr = L.threshold_from_review_budget(sv, float(best_params["soft_review_budget"]))
        rows.append(dict(model=name, category=cls, auroc_test=L.safe_auroc(yt, st), f1_test=L.prf_at_threshold(yt, st, thr)[2],
                         prevalence_test=prev[cls], f1_at_proditec_prevalence=L.f1_at_prevalence(yt, st, thr, PDT_PREV)))
PM = pd.DataFrame(rows)
agg = PM.groupby("model")[["auroc_test", "f1_test", "f1_at_proditec_prevalence"]].mean().reset_index()
display(PM.round(3)); display(agg.round(3))
save_table(PM, "R01_per_category_auroc_prevalence_matched")
save_table(agg, "R01_prevalence_matched_summary")
print("\nA detector's F1 depends on the positive rate. MVTec's 70 % vs Proditec's 34.5 % makes raw F1 incomparable across datasets; "
      "the column f1_at_proditec_prevalence re-weights the same decisions to Proditec's rate. AUROC is prevalence-free.")

In [ ]:
# ============================================================
# R1.7) Figures
# ============================================================
col = {"DiCE": "#3b82d4", "CFNOW": "#7c5cd8", "ACFX": "#d98a2b"}
fig, ax = plt.subplots(1, 2, figsize=WIDE_COL)
for m, g in tB.groupby("method"):
    ax[0].scatter(g.surrogate_pred, g.f1_val, s=14, color=col[m], label=f"{m} (val)", marker="o")
    ax[0].scatter(g.surrogate_pred, g.pooled_f1_test, s=14, color=col[m], marker="x", alpha=.8)
lim = [min(tB.surrogate_pred.min(), tB.pooled_f1_test.min(), tB.f1_val.min()) - .02, max(tB.surrogate_pred.max(), tB.f1_val.max()) + .02]
ax[0].plot(lim, lim, "k:", lw=.8); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel("Surrogate prediction"); ax[0].set_ylabel("Re-measured F1")
ax[0].set_title("o per-category (surrogate protocol)   x pooled (paper)"); ax[0].legend(frameon=False, fontsize=6, loc="lower right")
sub = SW.copy()
for m, c in (("HPO best", "#3b82d4"), ("reference (no HPO)", "#d98a2b")):
    g = sub[sub.model == m]; ax[1].plot(g.budget, g.f1_test, "-o", ms=2.5, color=c, label=m)
g = sw_hpo; ax[1].plot(g.budget, g.perfect, "k--", lw=.8, label="perfect ranker"); ax[1].plot(g.budget, g["random"], ":", color="grey", label="random ranker")
ax[1].axvline(float(best_params["soft_review_budget"]), color="r", lw=.6)
ax[1].set_xlabel("Review budget"); ax[1].set_ylabel("Held-out F1 (per-category)"); ax[1].legend(frameon=False, fontsize=6)
ax[1].set_title("HPO vs no-HPO at equal budget")
fig.tight_layout(); save_fig(fig, "R01_mvtec_protocol_matched_confirmation")

In [ ]:
# ============================================================
# R1.8) Freeze numbers
# ============================================================
merge_numbers("R01_mvtec_per_category", dict(
    smoke=SMOKE, n_candidates=int(len(tB)), seeds=list(SEEDS),
    hpo_best_logged=float(best_trial.value), hpo_best_rerun=float(a.f1_val),
    hpo_best_final_f1_rerun=tabA.set_index("category").f1_final_rerun.to_dict(),
    candidates_mean=allr.to_dict(), by_method=summ.set_index("method").round(4).to_dict(orient="index"),
    spearman_val=float(rho_val), spearman_test=float(rho_test),
    hpo_minus_reference_at_0p495=float(at(0.495)["HPO - reference"]),
    prevalence_matched=agg.set_index("model").round(4).to_dict(orient="index"),
))